In [1]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os


In [4]:
ROOT = '/kaggle/input/datasets/awsaf49/artifact-dataset'

CLASS_MAP = {
    # 8 real sources
    'afhq': 'real', 'celebahq': 'real', 'coco': 'real', 'ffhq': 'real',
    'imagenet': 'real', 'landscape': 'real', 'lsun': 'real', 'metfaces': 'real',

    # 13 GANs
    'big_gan': 'gan', 'cips': 'gan', 'cycle_gan': 'gan', 'gansformer': 'gan',
    'gau_gan': 'gan', 'pro_gan': 'gan', 'projected_gan': 'gan', 'star_gan': 'gan',
    'stylegan1': 'gan', 'stylegan2': 'gan', 'stylegan3': 'gan',
    'denoising_diffusion_gan': 'gan', 'diffusion_gan': 'gan',

    # 7 diffusion
    'ddpm': 'diffusion', 'glide': 'diffusion', 'latent_diffusion': 'diffusion',
    'palette': 'diffusion', 'stable_diffusion': 'diffusion',
    'vq_diffusion': 'diffusion', 'sfhq': 'diffusion',

    # 5 misc (not used)
    'face_synthetics': 'other', 'generative_inpainting': 'other',
    'lama': 'other', 'mat': 'other', 'taming_transformer': 'other',
}

# ---------- load every metadata.csv ----------
frames = []
for folder, cls in CLASS_MAP.items():
    csv_path = os.path.join(ROOT, folder, 'metadata.csv')
    if not os.path.exists(csv_path):
        print(f"  !! no metadata.csv in {folder}")
        continue
    d = pd.read_csv(csv_path)
    d['source'] = folder
    d['class'] = cls
    d['full_path'] = ROOT + '/' + folder + '/' + d['image_path']
    frames.append(d[['full_path', 'source', 'class']])

master = pd.concat(frames, ignore_index=True)
print(f"Total indexed: {len(master):,}\n")
print(master.groupby('class').size(), "\n")
print(master.groupby(['class', 'source']).size().to_string())

Total indexed: 2,496,738

class
diffusion      89243
gan          1248406
other         221705
real          937384
dtype: int64 

class      source                 
diffusion  ddpm                           896
           glide                        20903
           latent_diffusion             20000
           palette                       6000
           sfhq                         10000
           stable_diffusion             21444
           vq_diffusion                 10000
gan        big_gan                      10000
           cips                         11200
           cycle_gan                    15210
           denoising_diffusion_gan      10000
           diffusion_gan                15507
           gansformer                   10000
           gau_gan                       7000
           pro_gan                      40000
           projected_gan                12000
           star_gan                      9995
           stylegan1                    10000
      

In [5]:
SEED = 42
rng = np.random.RandomState(SEED)
TARGETS = {'real': 60000, 'gan': 30000, 'diffusion': 30000}

def sample_class(df, cls, total, rng):
    pool = df[df['class'] == cls]
    sources = sorted(pool['source'].unique())
    avail = {s: len(pool[pool['source'] == s]) for s in sources}

    quota = {s: 0 for s in sources}
    remaining = total
    active = set(sources)

    # Give everyone an equal share; anyone who can't fill it gets capped,
    # and their leftover is redistributed to the rest. Repeat until settled.
    while remaining > 0 and active:
        share = max(1, remaining // len(active))
        progressed = False
        for s in sorted(active):
            if remaining <= 0:
                break
            take = min(share, avail[s] - quota[s], remaining)
            if take > 0:
                quota[s] += take
                remaining -= take
                progressed = True
            if quota[s] >= avail[s]:
                active.discard(s)
        if not progressed:
            break

    picked = []
    for s, n in quota.items():
        if n == 0:
            continue
        sub = pool[pool['source'] == s]
        picked.append(sub.sample(n=n, random_state=rng))
    return pd.concat(picked, ignore_index=True)

parts = [sample_class(master, c, n, rng) for c, n in TARGETS.items()]
sampled = pd.concat(parts, ignore_index=True)

sampled['label'] = (sampled['class'] != 'real').astype(int)
sampled = sampled.sample(frac=1, random_state=SEED).reset_index(drop=True)

# 80/10/10, stratified by source so every architecture appears in every split
def add_split(df, rng):
    df = df.copy()
    df['split'] = 'train'
    for s in df['source'].unique():
        idx = df.index[df['source'] == s].to_numpy()
        rng.shuffle(idx)
        n = len(idx)
        n_tr, n_va = int(0.8 * n), int(0.1 * n)
        df.loc[idx[n_tr:n_tr + n_va], 'split'] = 'val'
        df.loc[idx[n_tr + n_va:], 'split'] = 'test'
    return df

sampled = add_split(sampled, np.random.RandomState(SEED))
sampled.to_csv('/kaggle/working/sampled_40k.csv', index=False)

print(f"Total: {len(sampled):,}\n")
print(sampled.groupby('class').size(), "\n")
print(sampled.groupby(['class', 'source']).size().to_string(), "\n")
print(sampled.groupby(['split', 'label']).size(), "\n")
print(sampled[['full_path', 'source', 'class', 'label', 'split']].head())

Total: 120,000

class
diffusion    30000
gan          30000
real         60000
dtype: int64 

class      source                 
diffusion  ddpm                        896
           glide                      4851
           latent_diffusion           4851
           palette                    4851
           sfhq                       4851
           stable_diffusion           4850
           vq_diffusion               4850
gan        big_gan                    2308
           cips                       2308
           cycle_gan                  2308
           denoising_diffusion_gan    2308
           diffusion_gan              2308
           gansformer                 2308
           gau_gan                    2308
           pro_gan                    2308
           projected_gan              2308
           star_gan                   2307
           stylegan1                  2307
           stylegan2                  2307
           stylegan3                  2307
real       

In [6]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class ArtiFactDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = np.array(Image.open(row['full_path']).convert("RGB"))
        if self.transform:
            img = self.transform(image=img)['image']
        label = torch.tensor(row['label'], dtype=torch.float32)
        return img, label

In [7]:
import albumentations as A
import numpy as np
from albumentations.pytorch import ToTensorV2

MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)
SIZE = 224

# Train: each degradation fires independently, at random strength
train_tf = A.Compose([
    A.HorizontalFlip(p=0.5),

    A.OneOf([
        A.ImageCompression(quality_range=(30, 95), p=1.0),
        A.GaussianBlur(blur_limit=(3, 7), sigma_limit=(0.3, 2.2), p=1.0),
        A.Downscale(scale_range=(0.25, 0.75),
                    interpolation_pair={'downscale': 0, 'upscale': 1}, p=1.0),
    ], p=0.7),

    A.OneOf([
        A.GaussNoise(std_range=(0.01, 0.12), p=1.0),
        A.ColorJitter(brightness=0.2, contrast=0.2,
                      saturation=0.2, hue=0.05, p=1.0),
    ], p=0.5),

    A.RandomResizedCrop(size=(SIZE, SIZE), scale=(0.7, 1.0), p=1.0),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])


In [8]:
import numpy as np
import torch
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)
SIZE = 224

# ---------- composable eval degradations ----------
DEGRADATIONS = {
    'jpeg':    lambda q:     A.ImageCompression(quality_range=(q, q), p=1.0),
    'blur':    lambda s:     A.GaussianBlur(blur_limit=(0, 0), sigma_limit=(s, s), p=1.0),
    'noise':   lambda s:     A.GaussNoise(std_range=(s, s), mean_range=(0, 0), p=1.0),
    'jitter':  lambda a:     A.ColorJitter(brightness=a, contrast=a,
                                           saturation=a, hue=0.0, p=1.0),
    'crop':    lambda f:     A.CenterCrop(int(200*f), int(200*f), p=1.0),
    'rescale': lambda s:     A.Downscale(scale_range=(s, s),
                                         interpolation_pair={'downscale': 0, 'upscale': 1},
                                         p=1.0),
}

def build_eval_tf(*steps):
    """
    build_eval_tf()                             -> clean
    build_eval_tf(('jpeg', 30))                 -> single
    build_eval_tf(('rescale', 0.5), ('jpeg', 50))  -> chained, in order
    """
    ops = []
    for name, param in steps:
        if name not in DEGRADATIONS:
            raise ValueError(f"unknown '{name}'. options: {list(DEGRADATIONS)}")
        ops.append(DEGRADATIONS[name](param))
    return A.Compose(ops + [
        A.Resize(SIZE, SIZE),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

# ---------- the three eval transforms used during training ----------
eval_clean_tf = build_eval_tf()
eval_hard_tf  = build_eval_tf(('rescale', 0.5), ('jpeg', 50), ('blur', 1.0))

# ---------- splits ----------
train_df = sampled[sampled['split'] == 'train']
val_df   = sampled[sampled['split'] == 'val']
test_df  = sampled[sampled['split'] == 'test']
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")

# ---------- datasets ----------
train_ds      = ArtiFactDataset(train_df, transform=train_tf)
val_clean_ds  = ArtiFactDataset(val_df,   transform=eval_clean_tf)
val_hard_ds   = ArtiFactDataset(val_df,   transform=eval_hard_tf)
test_ds       = ArtiFactDataset(test_df,  transform=eval_clean_tf)

# ---------- dataloaders ----------
BATCH = 64
NW = 2

train_dl     = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=NW, pin_memory=True, drop_last=True)
val_clean_dl = DataLoader(val_clean_ds, batch_size=BATCH, shuffle=False,
                          num_workers=NW, pin_memory=True)
val_hard_dl  = DataLoader(val_hard_ds, batch_size=BATCH, shuffle=False,
                          num_workers=NW, pin_memory=True)
test_dl      = DataLoader(test_ds, batch_size=BATCH, shuffle=False,
                          num_workers=NW, pin_memory=True)

# ---------- sanity check ----------
x, y = next(iter(train_dl))
print("\ntrain batch:", x.shape, x.dtype, "| labels:", y.shape, y.dtype)
print("range:", round(x.min().item(), 2), "to", round(x.max().item(), 2))
print("label mix:", y[:12].tolist())

xh, yh = next(iter(val_hard_dl))
print("hard batch:", xh.shape, "range:", round(xh.min().item(), 2), "to", round(xh.max().item(), 2))

train 95986 | val 11983 | test 12031

train batch: torch.Size([64, 3, 224, 224]) torch.float32 | labels: torch.Size([64]) torch.float32
range: -2.12 to 2.64
label mix: [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0]
hard batch: torch.Size([64, 3, 224, 224]) range: -2.12 to 2.64


In [9]:
import timm, torch

model = timm.create_model(
    'convnext_tiny',
    pretrained=True,
    num_classes=1,
    drop_path_rate=0.1,
)

print(model.default_cfg)
print(f"\nparams: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

{'url': '', 'hf_hub_id': 'timm/convnext_tiny.in12k_ft_in1k', 'architecture': 'convnext_tiny', 'tag': 'in12k_ft_in1k', 'custom_load': False, 'input_size': (3, 224, 224), 'test_input_size': (3, 288, 288), 'fixed_input_size': False, 'interpolation': 'bicubic', 'crop_pct': 0.95, 'test_crop_pct': 1.0, 'crop_mode': 'center', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'num_classes': 1000, 'pool_size': (7, 7), 'first_conv': 'stem.0', 'classifier': 'head.fc', 'license': 'apache-2.0'}

params: 27.8M


In [10]:
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)

model = model.to(device)

criterion = nn.BCEWithLogitsLoss()

device: cuda


In [11]:
import time
import torch
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR

# =====================================================================
# EVALUATION FUNCTION
# Reused for val_clean, val_hard, and later for the full degradation
# table. Takes any dataloader, returns (accuracy, average loss).
# =====================================================================
def evaluate(model, loader, device):
    model.eval()          # DropPath OFF, inference mode
    correct, total, loss_sum = 0, 0, 0.0

    # no_grad: don't build the computation graph. We're not training here
    # so gradients are useless. Saves memory and runs faster.
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            logits = model(x).squeeze(1)          # [64,1] -> [64]
            loss_sum += criterion(logits, y).item() * y.size(0)

            # model outputs a raw number (a "logit"), not a probability.
            # sigmoid maps it to 0..1, then 0.5 is the decision threshold.
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == y).sum().item()
            total   += y.size(0)

    return correct / total, loss_sum / total


# =====================================================================
# SETUP
# =====================================================================
EPOCHS = 15

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.05)

warmup = LinearLR(optimizer, start_factor=0.1, total_iters=len(train_dl))
cosine = CosineAnnealingLR(optimizer, T_max=EPOCHS - 1)

# ---- NEW: the AMP gradient scaler --------------------------------
# Why this exists: AMP runs the forward pass in float16, which is fast but
# has a small numeric range. Gradients are tiny (often ~1e-8) and anything
# below ~6e-8 rounds to ZERO in fp16, so the model silently stops learning.
# The scaler multiplies the loss by a big number (~65536) before backward
# so gradients come out big enough to survive fp16, then divides them back
# down before the weight update. Same math, no vanishing.
scaler = torch.amp.GradScaler('cuda')
# ------------------------------------------------------------------

history = []
best_score = 0.0


# =====================================================================
# TRAINING LOOP
# =====================================================================
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()                       # DropPath ON, training mode
    correct, total, loss_sum = 0, 0, 0.0

    for i, (x, y) in enumerate(train_dl):
        # DataLoader builds batches on CPU. Move them to GPU here.
        # non_blocking=True pairs with pin_memory=True to let the copy
        # overlap with computation instead of stalling.
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        # PyTorch ACCUMULATES gradients by default. Without this, batch 2's
        # gradients pile on top of batch 1's. set_to_none is a bit faster
        # than writing zeros.
        optimizer.zero_grad(set_to_none=True)

        # ---- NEW: autocast block -------------------------------------
        # Everything inside runs in float16 where it's safe. PyTorch picks
        # per-operation: convs and matmuls go fp16 (fast, the T4 has
        # dedicated hardware for it), precision-sensitive ops stay fp32.
        # ONLY the forward pass and loss go in here. Never backward(),
        # never step().
        with torch.amp.autocast('cuda'):
            logits = model(x).squeeze(1)     # [64,1] -> [64] to match y
            loss = criterion(logits, y)
        # --------------------------------------------------------------

        # ---- NEW: three scaler lines replace the usual two ------------
        # Plain version would be:
        #     loss.backward()
        #     optimizer.step()
        #
        # scale(loss).backward()  -> scale loss up, then compute grads
        # scaler.step(optimizer)  -> unscale grads, THEN call
        #                            optimizer.step() internally
        # scaler.update()         -> tune the multiplier for next batch
        #                            (shrinks it if anything overflowed)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        if epoch == 1:
            warmup.step() 
        # --------------------------------------------------------------

        # running stats. .item() pulls the value off the GPU. multiply by
        # batch size so the final average is weighted correctly.
        loss_sum += loss.item() * y.size(0)
        correct  += ((torch.sigmoid(logits) > 0.5).float() == y).sum().item()
        total    += y.size(0)

        if i % 100 == 0:
            print(f"  ep{epoch}| batch {i}/{len(train_dl)} | batch_loss {loss.item():.4f}")

    # LR schedule steps ONCE PER EPOCH (weights already updated ~500x above)
    
    if epoch > 1:
        cosine.step() 

    # ---------------- end-of-epoch evaluation ----------------
    train_acc  = correct / total
    train_loss = loss_sum / total
    vc_acc, vc_loss = evaluate(model, val_clean_dl, device)   # pristine
    vh_acc, vh_loss = evaluate(model, val_hard_dl,  device)   # degraded

    # Save on the AVERAGE of clean and hard, not clean alone. Saving on
    # clean would pick a model that's fragile under degradation, the exact
    # opposite of what this task is judged on.
    combined = (vc_acc + vh_acc) / 2

    history.append(dict(epoch=epoch, train_acc=train_acc, train_loss=train_loss,
                        val_clean=vc_acc, val_hard=vh_acc, combined=combined,
                        lr=optimizer.param_groups[0]['lr']))

    flag = ""
    if combined > best_score:
        best_score = combined
        torch.save(model.state_dict(), '/kaggle/working/best_model.pth')
        flag = "  <-- saved"

    print(f"epoch {epoch:2d} | train {train_acc:.4f} | "
          f"val_clean {vc_acc:.4f} | val_hard {vh_acc:.4f} | "
          f"{time.time()-t0:.0f}s{flag}\n")


hist_df = pd.DataFrame(history)
hist_df.to_csv('/kaggle/working/history.csv', index=False)
print(hist_df.to_string(index=False))

/tmp/ipykernel_58/1806108439.py:103: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warmup.step()


  ep1| batch 0/1499 | batch_loss 0.8277
  ep1| batch 100/1499 | batch_loss 0.5783
  ep1| batch 200/1499 | batch_loss 0.6348
  ep1| batch 300/1499 | batch_loss 0.5930
  ep1| batch 400/1499 | batch_loss 0.5335
  ep1| batch 500/1499 | batch_loss 0.5748
  ep1| batch 600/1499 | batch_loss 0.5824
  ep1| batch 700/1499 | batch_loss 0.5569
  ep1| batch 800/1499 | batch_loss 0.6566
  ep1| batch 900/1499 | batch_loss 0.6297
  ep1| batch 1000/1499 | batch_loss 0.5898
  ep1| batch 1100/1499 | batch_loss 0.5632
  ep1| batch 1200/1499 | batch_loss 0.5233
  ep1| batch 1300/1499 | batch_loss 0.5633
  ep1| batch 1400/1499 | batch_loss 0.5693
epoch  1 | train 0.6736 | val_clean 0.7099 | val_hard 0.6281 | 837s  <-- saved

  ep2| batch 0/1499 | batch_loss 0.5157
  ep2| batch 100/1499 | batch_loss 0.6183
  ep2| batch 200/1499 | batch_loss 0.5753
  ep2| batch 300/1499 | batch_loss 0.6174
  ep2| batch 400/1499 | batch_loss 0.6902
  ep2| batch 500/1499 | batch_loss 0.5860
  ep2| batch 600/1499 | batch_loss 0.

In [12]:
model.load_state_dict(torch.load('/kaggle/working/best_model.pth'))
model.eval()

rows = []
for src in sorted(val_df['source'].unique()):
    sub = val_df[val_df['source'] == src]
    dl = DataLoader(ArtiFactDataset(sub, transform=eval_clean_tf),
                    batch_size=64, num_workers=2)
    acc, _ = evaluate(model, dl, device)
    cls = sub['class'].iloc[0]
    rows.append((cls, src, len(sub), round(acc, 3)))

print(pd.DataFrame(rows, columns=['class','source','n','acc'])
        .sort_values('acc').to_string(index=False))

    class                  source   n   acc
      gan           diffusion_gan 230 0.348
      gan                 pro_gan 230 0.483
      gan               stylegan3 230 0.496
      gan               stylegan2 230 0.561
      gan denoising_diffusion_gan 230 0.578
      gan               cycle_gan 230 0.604
      gan                 gau_gan 230 0.630
diffusion                   glide 485 0.631
      gan           projected_gan 230 0.657
diffusion            vq_diffusion 485 0.660
     real                    lsun 905 0.741
diffusion                    ddpm  89 0.742
diffusion        latent_diffusion 485 0.748
     real                metfaces 133 0.767
      gan                 big_gan 230 0.774
     real                imagenet 905 0.786
     real                    coco 905 0.836
diffusion        stable_diffusion 485 0.841
     real                    ffhq 905 0.843
      gan               stylegan1 230 0.865
diffusion                 palette 485 0.907
     real                celebah

In [13]:
from sklearn.metrics import roc_auc_score
import numpy as np

# load the best checkpoint
model.load_state_dict(torch.load('/kaggle/working/best_model.pth'))
model.eval()

# collect probabilities and true labels over the whole val set
probs, labels = [], []
with torch.no_grad():
    for x, y in val_clean_dl:
        p = torch.sigmoid(model(x.to(device)).squeeze(1))
        probs.append(p.cpu())
        labels.append(y)

probs  = torch.cat(probs).numpy()
labels = torch.cat(labels).numpy()

print("AUC:", round(roc_auc_score(labels, probs), 4))
print("mean prob, real images:", round(probs[labels==0].mean(), 3))
print("mean prob, fake images:", round(probs[labels==1].mean(), 3))
print()

# accuracy at different cutoffs
for t in [0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]:
    pred = (probs > t).astype(float)
    acc       = (pred == labels).mean()
    real_acc  = (pred[labels==0] == 0).mean()
    fake_acc  = (pred[labels==1] == 1).mean()
    print(f"thresh {t:.2f} | overall {acc:.3f} | real {real_acc:.3f} | fake {fake_acc:.3f}")

AUC: 0.8764
mean prob, real images: 0.203
mean prob, fake images: 0.72

thresh 0.15 | overall 0.765 | real 0.644 | fake 0.886
thresh 0.20 | overall 0.778 | real 0.697 | fake 0.859
thresh 0.25 | overall 0.786 | real 0.737 | fake 0.835
thresh 0.30 | overall 0.793 | real 0.771 | fake 0.814
thresh 0.35 | overall 0.795 | real 0.796 | fake 0.794
thresh 0.40 | overall 0.796 | real 0.817 | fake 0.775
thresh 0.45 | overall 0.795 | real 0.834 | fake 0.756
thresh 0.50 | overall 0.795 | real 0.853 | fake 0.738


In [14]:
hist_df.to_csv('/kaggle/working/run3_40k.csv', index=False)